# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection established.")
print("Using March 2026 development data.")

DuckDB connection established.
Using March 2026 development data.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
# Build the March 2026 feature table for signal auditing

audit_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(
            NULLIF(gsc_avg_position, 0)
        ) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Rows:", len(audit_df))
display(audit_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_volatility
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.888929,2.119233
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.240351,1.376604
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,7.061594,3.686090
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.141844,6.155424,3.892530
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.343567,14.439705


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal verdicts

**Signal #1 — Impressions / volume: CONFIRMED**

Higher-impression pages show higher median CTR. Pages with fewer than 500 impressions have a median CTR of 0%, while the 500–1,999 and 2,000+ groups have median CTRs of 0.155% and 0.207%. This supports using search volume as an important context signal and treating very low-volume observations cautiously.

**Signal #2 — Position vs CTR: CONFIRMED**

CTR decreases consistently as average search position worsens. In the 500+ impression flag-linked test, median CTR falls from 0.263% for positions 1–3 to 0.071% for positions 20+. This supports the assumption behind the CTR-fix signal.

**Signal #3 — Position volatility: MIXED**

CTR decreases across volatility quartiles, but volatility is also strongly associated with average position and impressions. High-volatility pages have a median position of 30.42 and median impressions of 71, while low-volatility pages have a median position of 4.63 and median impressions of 1,100. The audit therefore supports volatility as a potentially useful descriptive signal, but does not isolate it as an independent explanation for CTR differences.

In [3]:
distribution_summary = audit_df[
    [
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "position_volatility"
    ]
].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

display(distribution_summary)

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
impressions,176738.0,1587.986675,5431.337724,1.000000,20.000000,173.000000,1039.000000,3930.000000,7238.150000,21799.780000,617124.000000
clicks,176738.0,4.650002,26.722649,0.000000,0.000000,0.000000,2.000000,10.000000,22.000000,73.000000,5668.000000
ctr,176738.0,0.459397,3.775992,0.000000,0.000000,0.000000,0.215796,0.615385,1.086957,5.882353,100.000000
avg_position,175304.0,17.050555,18.333942,0.101639,5.500000,9.000000,22.000000,42.995319,59.521719,81.137720,309.000000
position_volatility,161557.0,9.292292,9.955904,0.000000,2.407726,5.382745,13.273467,23.299599,29.072908,40.869567,238.294985


In [4]:
# Three mini-tests for the signals we may use in the clustering analysis

# ---------------------------------------------------------
# Signal #1: Search volume / impressions
# ---------------------------------------------------------

volume_test = audit_df.copy()

volume_test["volume_bucket"] = pd.cut(
    volume_test["impressions"],
    bins=[-1, 99, 499, 1999, float("inf")],
    labels=["<100", "100-499", "500-1999", "2000+"]
)

volume_result = (
    volume_test
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("impressions", "size"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

print("SIGNAL #1 — IMPRESSIONS / VOLUME")
display(volume_result)


# ---------------------------------------------------------
# Signal #2: Average position vs CTR
# ---------------------------------------------------------

position_test = audit_df[
    audit_df["avg_position"].notna()
].copy()

position_test["position_bucket"] = pd.cut(
    position_test["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "20+"]
)

position_result = (
    position_test
    .groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("SIGNAL #2 — POSITION vs CTR")
display(position_result)


# ---------------------------------------------------------
# Signal #3: Position volatility vs CTR
# ---------------------------------------------------------

volatility_test = audit_df[
    audit_df["position_volatility"].notna()
].copy()

volatility_test["volatility_bucket"] = pd.qcut(
    volatility_test["position_volatility"],
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"],
    duplicates="drop"
)

volatility_result = (
    volatility_test
    .groupby("volatility_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

print("SIGNAL #3 — POSITION VOLATILITY")
display(volatility_result)

SIGNAL #1 — IMPRESSIONS / VOLUME


,volume_bucket,n,median_ctr,median_position
0,<100,75297,0.000000,9.000000
1,100-499,39517,0.000000,12.983691
2,500-1999,32047,0.155039,8.281417
3,2000+,29877,0.206954,6.353468


SIGNAL #2 — POSITION vs CTR


,position_bucket,n,median_ctr,mean_ctr
0,1-3,13136,0.096246,1.100994
1,4-10,81619,0.000000,0.514867
2,11-20,32548,0.000000,0.330347
3,20+,48001,0.000000,0.192806


SIGNAL #3 — POSITION VOLATILITY


,volatility_bucket,n,median_ctr,median_position,median_impressions
0,Low,40390,0.139470,4.630740,1100.0
1,Medium-Low,40389,0.023941,6.872727,411.0
2,Medium-High,40389,0.000000,13.353333,357.0
3,High,40389,0.000000,30.422587,71.0


### Flag-linked test verdict

**Signal tested:** CTR vs average search position  
**FlyRank flag logic:** CTR-fix  
**Minimum visibility:** 500 impressions  
**Verdict: CONFIRMED**

The flag-linked test shows a consistent decrease in CTR as search position worsens. Among pages with at least 500 impressions, median CTR decreases from 0.263% at positions 1–3 to 0.071% at positions 20+. This supports the underlying directional assumption used by the CTR-fix logic.

This is evidence of an observed relationship, not proof that changing a page's title, snippet, or content will cause CTR to increase.

In [5]:
# Flag-linked test: CTR-fix logic
# Question: among pages with meaningful visibility,
# does CTR remain lower in worse position tiers?

flag_test = audit_df[
    (audit_df["impressions"] >= 500) &
    (audit_df["avg_position"].notna()) &
    (audit_df["ctr"].notna())
].copy()

flag_test["position_bucket"] = pd.cut(
    flag_test["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "20+"]
)

flag_result = (
    flag_test
    .groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

display(flag_result)

print("Flag-linked signal: CTR vs position")
print("Minimum visibility threshold: 500 impressions")

,position_bucket,n,median_ctr,mean_ctr,median_impressions
0,1-3,7008,0.263363,0.378970,2616.0
1,4-10,31873,0.217282,0.321327,2012.0
2,11-20,11836,0.162075,0.261551,1354.0
3,20+,11207,0.071480,0.134751,1941.0


Flag-linked signal: CTR vs position
Minimum visibility threshold: 500 impressions


### Practical takeaway

Content teams should prioritize pages with meaningful search visibility and evaluate CTR in the context of search position rather than using a single CTR threshold. The audit supports using impressions and position as useful signals for prioritization, while position volatility should be treated as contextual rather than as independent evidence of a performance problem. Any recommended action should still be reviewed against search intent, SERP context, and page-level information before implementation.

### Self-check

- [x] The notebook sections are filled with both analysis and supporting code.
- [x] The notebook runs from top to bottom without errors.
- [x] Key field distributions were inspected before choosing signal tests.
- [x] Three signals were tested with visible bucket tables and sample counts (`n`).
- [x] Each signal has an explicit verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.
- [x] At least one signal was linked to a real FlyRank flag assumption and tested separately.
- [x] The practical takeaway uses observed/ measured language and avoids causal claims.
- [x] No client names, URLs, private queries, future-window data, or label-derived inputs were used.
- [x] The notebook is saved under `work/notebooks/w04_signal_audit.ipynb`.